# QMCPy Performance Optimizations Demo

Sou-Cheng T. Choi

Illinois Institute of Technology and SouLab LLC.

Modification date: 9/12/2026

Creation date: 8/14/2026

For reproducibility, this notebook was run with:
- Python 3.13.13, NumPy 2.5.0, SciPy 1.17.1, QMCPy 2.4, PyTorch (for the multitask kernel section)
- OS: macOS 15.6.1

This notebook benchmarks a set of small, targeted optimizations applied to QMCPy's internals: replacing generic `scipy.stats` distribution-object calls with the underlying `scipy.special` C-level functions, replacing `np.einsum` calls (run with its default, non-BLAS-routed contraction path) with `@`/`matmul`, and (Section 8) replacing a nested Python double loop with NumPy broadcasting.

**Why these are fast:**

- `scipy.stats.norm.ppf(x)` is a method on a generic `rv_continuous` distribution object. Even though it ultimately calls the same underlying Cephes routine, it first pays for loc/scale handling, support-bounds validation, and edge-case masking. `scipy.special.ndtri(x)` (and `ndtr(x)` for the forward CDF) call that routine directly.
- `np.einsum(subscripts, A, B)` is a general-purpose tensor-contraction *interpreter*. Unless called with `optimize=True`, it does not automatically recognize that a given contraction is really just a matrix multiply, so it doesn't route through BLAS's `GEMM` kernel the way `@`/`np.matmul` does. For contractions that *are* matrix multiplies in disguise, this can be a 10-50x difference.
- A nested `for i / for j` Python loop over an `n x n` grid does `n^2` individual Python-level operations; the equivalent NumPy broadcast (`t[:, None] + t[None, :]`, `np.minimum.outer(t, t)`, ...) does the same `n^2` work as compiled, vectorized C loops instead.

Each section below defines a small "legacy" function using the old approach, times it against the equivalent QMCPy call (which now uses the fast approach), and reports the speedup. All outputs are verified numerically identical (up to floating-point rounding) before timing.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMCSoftware/blob/develop/demos/performance_optimizations_demo.ipynb)

In [1]:
# @title Execute this cell to install dependencies
try:
  import google.colab
  IN_COLAB = True
except ImportError:
  IN_COLAB = False
if IN_COLAB:
  !pip install -q qmcpy


In [2]:
import time
import numpy as np
from scipy.stats import norm
from scipy.stats import t as t_dist
from scipy.special import ndtri, ndtr, stdtrit

from qmcpy import BrownianMotion, DigitalNetB2

rng = np.random.default_rng(7)

def bench(f, target_time=0.05, max_reps=5000):
    f()  # warm up
    reps = 5
    while True:
        times = np.empty(reps)
        for i in range(reps):
            t0 = time.perf_counter()
            result = f()
            times[i] = time.perf_counter() - t0
        # Report min-over-reps once total measured time clears target_time: for
        # sub-millisecond calls, a few reps of a mean-based timer are dominated by
        # scheduler/GC jitter rather than true cost, which can flip which side "wins".
        if times.sum() >= target_time or reps >= max_reps:
            return result, times.min()
        reps *= 4

results_table = []

def report(name, t_old, t_new, max_diff):
    assert max_diff < 1e-8, f'{name}: old and new results diverge (max diff {max_diff:.2e})'
    speedup = t_old / t_new
    results_table.append((name, t_old, t_new, speedup, max_diff))
    print(f'{name}')
    print(f'  old: {t_old:.5f} s   new: {t_new:.5f} s   speedup: {speedup:.1f}x   max diff: {max_diff:.2e}')


## 1. Gaussian PCA Transform (`qmcpy/true_measure/gaussian.py`)

This is the code path used by default (`decomp_type="PCA"`) by `Gaussian`, `BrownianMotion`, and `GeometricBrownianMotion` -- so it's exercised any time `demos/GBM/gbm_demo.ipynb` samples GBM paths with a Sobol', Lattice, or Halton sampler. `A` below stands in for the cached PCA factor matrix.

In [3]:
n, d = 2**14, 252
x = rng.random((n, d))
A = rng.standard_normal((d, d))
mu = np.zeros(d)

def legacy_gaussian_transform():
    return mu + np.einsum('...ij,kj->...ik', norm.ppf(x), A)

def fast_gaussian_transform():
    out = ndtri(x) @ A.T
    out += mu
    return out

r_old, t_old = bench(legacy_gaussian_transform)
r_new, t_new = bench(fast_gaussian_transform)
report('Gaussian PCA transform (gaussian.py)', t_old, t_new, np.abs(r_old - r_new).max())


Gaussian PCA transform (gaussian.py)
  old: 0.23724 s   new: 0.03469 s   speedup: 6.8x   max diff: 1.21e-13


## 2. Brownian Bridge Transform (`qmcpy/true_measure/brownian_motion.py`)

Used whenever `decomp_type="BrownianBridge"` -- the construction showcased in `demos/brownian_bridge.ipynb`. We compare `BrownianMotion`'s actual bridge transform against a copy of the same code with `ndtri` swapped back for `norm.ppf`.

`_bridge_transform` builds each dimension from up to two earlier ones (Owen's bisection construction), which used to run as an explicit `for j in range(d)` Python loop. Since points at the same bisection depth never depend on each other, they can be grouped by depth (precomputed once, at construction time) and each depth level updated with one vectorized NumPy op instead -- turning `d` Python iterations into `O(log d)`. The output is bit-identical; only the grouping of independent work changed. (A first attempt batched levels along the *last* axis of the `(n_samples, d)` array and came out *slower* than the plain loop -- NumPy's advanced indexing on a strided axis has real overhead. Transposing so the dimension axis is contiguous first fixed that.)

This batching only pays for itself once a depth level is large enough to amortize the moveaxis/reshape/fancy-indexing setup it costs. Two configurations have no level that ever gets large: small `d` (the common case in unit tests and short-horizon simulations -- e.g. `d=16`'s levels are only `[1,1,2,4,8]`), and an already-increasing `monitoring_times` grid with `bridge_vdc_gray_ordering=False` (the bisection tree degenerates into a linear chain of `d` singleton levels, regardless of `d`). Both used to be *slower* than the original loop -- up to ~4.7x at `d=2` and ~4x on the degenerate grid at `d=256` -- which was most of a real "Unit Tests went from ~7min to 9-11min" CI regression on this branch. `_bridge_transform` now checks `_bridge_max_level_size` (and, per level, its own size) and falls back to the plain scalar update whenever batching wouldn't help, so both cases are back to matching the pre-batching cost exactly, while the genuinely large, tree-shaped case below keeps a real win.

In [4]:
d, n_paths = 256, 2**12
bm = BrownianMotion(DigitalNetB2(d, seed=7), decomp_type='BrownianBridge')
u = rng.random((n_paths, d))

def legacy_bridge_transform():
    z = norm.ppf(u)
    w = bm._bridge_transform(z)
    paths = bm.drift_time_vec_plus_init + np.sqrt(bm.diffusion) * w
    return paths[..., bm._output_order]

def fast_bridge_transform():
    return bm._transform(u)

r_old, t_old = bench(legacy_bridge_transform)
r_new, t_new = bench(fast_bridge_transform)
report('Brownian bridge transform (brownian_motion.py)', t_old, t_new, np.abs(r_old - r_new).max())


Brownian bridge transform (brownian_motion.py)
  old: 0.02220 s   new: 0.01463 s   speedup: 1.5x   max diff: 0.00e+00


**Isolating the loop-vectorization from the `ndtri` swap:** the comparison above holds the bridge construction fixed and only swaps `norm.ppf` for `ndtri`, so it doesn't show the level-batching change on its own -- both sides call the same (already-optimized) `_bridge_transform`. Here's the original per-dimension Python loop against the current level-batched version directly, both fed the same pre-computed `z`, isolating just that change.

In [5]:
def legacy_bridge_loop(z):
    left, right = bm._bridge_left, bm._bridge_right
    a, b, w = bm._bridge_a, bm._bridge_b, bm._bridge_w
    paths = np.empty(z.shape[:-1] + (bm.d,))
    for j in range(bm.d):
        paths[..., j] = w[j] * z[..., j]
        if left[j] >= 0:
            paths[..., j] += a[j] * paths[..., left[j]]
        if right[j] >= 0:
            paths[..., j] += b[j] * paths[..., right[j]]
    return paths[..., bm._increasing_order]

z = ndtri(u)

def legacy_bridge_loop_call():
    return legacy_bridge_loop(z)

def fast_bridge_level_batched():
    return bm._bridge_transform(z)

r_old, t_old = bench(legacy_bridge_loop_call)
r_new, t_new = bench(fast_bridge_level_batched)
report('Brownian bridge loop -> level-batched (brownian_motion.py)', t_old, t_new, np.abs(r_old - r_new).max())


Brownian bridge loop -> level-batched (brownian_motion.py)
  old: 0.01315 s   new: 0.00492 s   speedup: 2.7x   max diff: 0.00e+00


## 3. Gaussian Copula Density (`qmcpy/true_measure/gaussian_copula.py`)

Used by `GaussianCopula._weight()`, showcased in `demos/copula_examples.ipynb`. This combines *both* techniques: `norm.ppf` -> `ndtri`, and a 3-operand quadratic-form `einsum` -> a matmul-then-reduce.

In [6]:
d, n = 40, 2**14
M = rng.standard_normal((d, d)); M = M @ M.T  # stand-in for corr_inv_minus_eye
u = rng.random((n, d))
logdet = 0.0

def legacy_copula_quad():
    z = norm.ppf(u)
    quad = np.einsum('...i,ij,...j->...', z, M, z)
    return -0.5 * logdet - 0.5 * quad

def fast_copula_quad():
    z = ndtri(u)
    quad = ((z @ M) * z).sum(-1)
    return -0.5 * logdet - 0.5 * quad

r_old, t_old = bench(legacy_copula_quad)
r_new, t_new = bench(fast_copula_quad)
report('Gaussian copula log-density (gaussian_copula.py)', t_old, t_new, np.abs(r_old - r_new).max())


Gaussian copula log-density (gaussian_copula.py)
  old: 0.03814 s   new: 0.00534 s   speedup: 7.1x   max diff: 7.50e-12


## 4. Multitask Kernel Matrix (`qmcpy/kernel/multitask_kernel.py`)

`KernelMultiTask.taskmat` builds a batched task-covariance matrix for multi-output/multi-fidelity Bayesian cubature. It previously used `einsum("...ij,...kj->...ik", ...)`; now it uses a batched `matmul`.

In [7]:
batch, num_tasks, rank = 2**10, 8, 8
factor = rng.standard_normal((batch, num_tasks, rank))

def legacy_taskmat():
    return np.einsum('...ij,...kj->...ik', factor, factor)

def fast_taskmat():
    return np.matmul(factor, np.swapaxes(factor, -1, -2))

r_old, t_old = bench(legacy_taskmat)
r_new, t_new = bench(fast_taskmat)
report('Multitask kernel taskmat (multitask_kernel.py)', t_old, t_new, np.abs(r_old - r_new).max())


Multitask kernel taskmat (multitask_kernel.py)
  old: 0.00031 s   new: 0.00009 s   speedup: 3.6x   max diff: 7.11e-15


## 5. Bayesian Logistic Regression Integrand (`qmcpy/integrand/bayesian_lr_coeffs.py`)

`BayesianLRCoeffs.g(x)` evaluates the log-likelihood integrand at each sampled coefficient vector; showcased in `demos/vectorized_qmc_bayes.ipynb`. The `einsum("...j,ij->...i", x, feature_array)` step is really `x @ feature_array.T`.

In [8]:
n, n_coeffs, n_obs = 2**14, 30, 5
x = rng.standard_normal((n, n_coeffs))
feature_array = rng.standard_normal((n_obs, n_coeffs))

def legacy_bayesian_lr():
    return np.einsum('...j,ij->...i', x, feature_array)

def fast_bayesian_lr():
    return x @ feature_array.T

r_old, t_old = bench(legacy_bayesian_lr)
r_new, t_new = bench(fast_bayesian_lr)
report('Bayesian LR feature projection (bayesian_lr_coeffs.py)', t_old, t_new, np.abs(r_old - r_new).max())


Bayesian LR feature projection (bayesian_lr_coeffs.py)
  old: 0.00033 s   new: 0.00019 s   speedup: 1.7x   max diff: 1.07e-14


## 6. Student-t Quantile (`qmcpy/true_measure/student_t.py`, `student_t_copula.py`)

`StudentT` and `StudentTCopula` build multivariate Student-t samples via a sequence of univariate conditional quantiles, each previously computed with `scipy.stats.t.ppf`. `scipy.special.stdtrit` is the direct-call equivalent -- note the argument order flips from `t.ppf(q, df)` to `stdtrit(df, q)`.

Unlike `ndtri`, `stdtrit` is not a simple closed-form transform -- Cephes solves for it iteratively, so its own compute cost grows with array size and increasingly dominates the fixed `rv_continuous` overhead being removed. The win here is therefore much more size-dependent than in Sections 1-3: large for a handful of values, modest for a full batch of QMC points.

In [9]:
df = 6.0
n_paths = 2**10
p = rng.random(n_paths)

def legacy_t_quantile():
    return t_dist.ppf(p, df=df)

def fast_t_quantile():
    return stdtrit(df, p)

r_old, t_old = bench(legacy_t_quantile)
r_new, t_new = bench(fast_t_quantile)
report('Student-t quantile (student_t_copula.py)', t_old, t_new, np.abs(r_old - r_new).max())


Student-t quantile (student_t_copula.py)
  old: 0.00017 s   new: 0.00013 s   speedup: 1.4x   max diff: 0.00e+00


## 7. Black-Scholes Exact Price (`qmcpy/integrand/financial_option.py`)

`FinancialOption.get_exact_value()` prices European/Asian options with closed-form Black-Scholes-type formulas -- each a handful of *scalar* `norm.cdf` evaluations, called once per pricing request rather than once per QMC sample. This is effectively the opposite regime from Section 5's Bayesian LR case: the workload is tiny either way, but here `norm.cdf`'s `rv_continuous` overhead is the *entire* cost (there's no BLAS/vectorization to fall back on), so removing it wins by orders of magnitude rather than being noise-dominated.

In [10]:
d1, d2 = 0.35, 0.15  # stand-in for the two scalar log-moneyness terms in get_exact_value

def legacy_bs_cdf():
    return norm.cdf(d1) - norm.cdf(d2)

def fast_bs_cdf():
    return ndtr(d1) - ndtr(d2)

r_old, t_old = bench(legacy_bs_cdf)
r_new, t_new = bench(fast_bs_cdf)
report('Black-Scholes normal CDF (financial_option.py)', t_old, t_new, np.abs(r_old - r_new).max())


Black-Scholes normal CDF (financial_option.py)
  old: 0.00004 s   new: 0.00000 s   speedup: 468.0x   max diff: 0.00e+00


## 8. GBM Covariance Matrix (`qmcpy/true_measure/geometric_brownian_motion.py`)

`GeometricBrownianMotion.covariance_gbm` builds the `n x n` covariance matrix once per instance, where `n` is the number of time steps (e.g. 252 for a typical trading-year GBM, the default used throughout `demos/GBM/gbm_demo.ipynb`). It already had a broadcasted formula for `n<=200`; above that threshold it fell back to a nested Python `for i / for j` loop, with a comment claiming this was for "memory efficiency." That claim didn't hold up: both branches already allocate a full `n x n` array (the loop version explicitly via `np.zeros((n, n))`), so there was no memory actually being saved -- just a real runtime cost, and `n=252` fell on the slow side of the threshold. The loop is gone; the broadcasted formula now runs for every `n`.

In [11]:
def legacy_gbm_covariance(t, S0_sq, mu, diffusion):
    n = len(t)
    cov_matrix = np.zeros((n, n))
    exp_mu_t = np.exp(mu * t)
    exp_diff_t = np.exp(diffusion * t)
    for i in range(n):
        cov_matrix[i, i] = S0_sq * exp_mu_t[i] ** 2 * (exp_diff_t[i] - 1)
        for j in range(i + 1, n):
            t_min_ij = min(t[i], t[j])
            cov_ij = S0_sq * exp_mu_t[i] * exp_mu_t[j] * (np.exp(diffusion * t_min_ij) - 1)
            cov_matrix[i, j] = cov_ij
            cov_matrix[j, i] = cov_ij
    return cov_matrix

def fast_gbm_covariance(t, S0_sq, mu, diffusion):
    t_sum = t[:, None] + t[None, :]
    t_min = np.minimum.outer(t, t)
    return S0_sq * np.exp(mu * t_sum) * (np.exp(diffusion * t_min) - 1)

n_steps = 252  # the default used throughout demos/GBM/gbm_demo.ipynb -- and >200, the old threshold
t = np.linspace(1 / n_steps, 1.0, n_steps)
S0_sq, mu, diffusion = 100.0**2, 0.05, 0.2**2

def legacy_gbm_cov_call():
    return legacy_gbm_covariance(t, S0_sq, mu, diffusion)

def fast_gbm_cov_call():
    return fast_gbm_covariance(t, S0_sq, mu, diffusion)

r_old, t_old = bench(legacy_gbm_cov_call)
r_new, t_new = bench(fast_gbm_cov_call)
report('GBM covariance matrix (geometric_brownian_motion.py)', t_old, t_new, np.abs(r_old - r_new).max())


GBM covariance matrix (geometric_brownian_motion.py)
  old: 0.01314 s   new: 0.00026 s   speedup: 50.4x   max diff: 1.71e-13


## Summary

In [12]:
import pandas as pd
summary_df = pd.DataFrame(
    results_table,
    columns=['Component', 'Old (s)', 'New (s)', 'Speedup', 'Max Abs Diff'],
)
sorted_summary_df = summary_df.sort_values(by='Speedup', ascending=False)
sorted_summary_df.round(6)


,Component,Old (s),New (s),Speedup,Max Abs Diff
7,Black-Scholes normal CDF (financial_option.py),0.000039,0.000000,467.994382,0.0
8,GBM covariance matrix (geometric_brownian_moti...,0.013136,0.000261,50.425143,0.0
3,Gaussian copula log-density (gaussian_copula.py),0.038139,0.005343,7.138014,0.0
0,Gaussian PCA transform (gaussian.py),0.237239,0.034691,6.838687,0.0
4,Multitask kernel taskmat (multitask_kernel.py),0.000307,0.000086,3.589775,0.0
2,Brownian bridge loop -> level-batched (brownia...,0.013155,0.004918,2.674633,0.0
5,Bayesian LR feature projection (bayesian_lr_co...,0.000327,0.000193,1.692178,0.0
1,Brownian bridge transform (brownian_motion.py),0.022201,0.014630,1.517504,0.0
6,Student-t quantile (student_t_copula.py),0.000173,0.000127,1.362357,0.0


**Where these speedups show up in existing demos**, next time each is re-run:

- `demos/GBM/gbm_demo.ipynb`, `demos/GBM/gbm_examples.ipynb` -- GeometricBrownianMotion and BrownianMotion path generation (Sections 1, 2, and 8 above)
- `demos/brownian_bridge.ipynb` -- BrownianMotion with `decomp_type="BrownianBridge"` (Section 2)
- `demos/copula_examples.ipynb`, `demos/product_measure.ipynb` -- GaussianCopula (Section 3) and StudentTCopula (Section 6)
- `demos/vectorized_qmc.ipynb`, `demos/vectorized_qmc_bayes.ipynb` -- BayesianLRCoeffs (Section 5)
- `demos/pricing_options.ipynb`, `demos/asian-option-mlqmc.ipynb`, `demos/control_variates.ipynb` -- FinancialOption exact-value pricing (Section 7)

No existing demo currently exercises `KernelMultiTask` (Section 4); it is used internally by multi-output/multi-fidelity Bayesian cubature stopping criteria.

## Appendix: Other `scipy.stats` -> `scipy.special` swaps

The overhead-elimination trick above generalizes: any `scipy.stats.<dist>.{ppf,cdf,logcdf,...}` call has a `scipy.special` counterpart that skips the generic `rv_continuous`/`rv_discrete` dispatch (loc/scale handling, support-bounds validation, edge-case masking) and calls the underlying compiled routine directly.

| Instead of (`scipy.stats`, slow) | Use (`scipy.special`, fast) | Purpose | Used in QMCPy? |
|---|---|---|---|
| `norm.cdf` | `ndtr` | forward normal CDF | Yes -- `gaussian_copula.py`, `financial_option.py`, several stopping criteria (Sections 3, 7 above) |
| `norm.ppf` | `ndtri` | inverse normal CDF | Yes -- `gaussian.py`, `brownian_motion.py`, `johnsons_su.py`, several stopping criteria (Sections 1-2 above) |
| `norm.logcdf` | `log_ndtr` | log of normal CDF, numerically stable in the tails | Not currently used |
| `t.cdf` | `stdtr` | Student-t CDF | Yes -- `student_t_copula.py` |
| `t.ppf` | `stdtrit` | Student-t quantile (argument order flips to `stdtrit(df, p)`) | Yes -- `student_t.py`, `student_t_copula.py`, `cub_qmc_rep_student_t.py` (Section 6 above) |
| `chi2.cdf` | `chdtr` | chi-squared CDF | Not currently used |
| `chi2.ppf` | `chdtri` (note: `chdtri(df, 1 - q)`, since it inverts the *complemented* CDF -- `chdtriv` is a different function, the inverse with respect to degrees of freedom, not a quantile) | chi-squared quantile | Not currently used |
| `gamma.cdf` | `gammainc` (regularized lower incomplete gamma) | gamma CDF | Not currently used |
| `gamma.ppf` | `gammaincinv` (note: NOT `gammainccinv`, which inverts the *upper* incomplete gamma -- that's `gamma.isf`, not `gamma.ppf`) | gamma quantile | Not currently used |
| `beta.ppf` / `.cdf` | `betaincinv` / `betainc` | beta quantile/CDF | Not applicable -- `kumaraswamy.py` looks similar but is not a Beta distribution; it has its own closed-form quantile, `(1-(1-x)**(1/b))**(1/a)`, already faster than any `scipy` Beta call |
| `poisson.cdf` | `pdtr` | Poisson CDF | Not currently used |
| `binom.cdf` | `bdtr` | binomial CDF | Not currently used |
| `f.cdf` / `.ppf` | `fdtr` / `fdtri` | F-distribution CDF/quantile | Not currently used |

Sections 1-3, 6, and 7 above benchmark the rows already used in QMCPy; the rest of the table is included for reference should a future distribution need one of these. (The chi2/gamma quantile rows above were originally wrong in an earlier draft of this table -- `chdtriv`/`gammainccinv` are not equivalents of `.ppf` at all; verified numerically, e.g. `gamma.ppf(0.1, 2) == gammaincinv(2, 0.1) == 0.531812`, not `gammainccinv(2, 0.1) == 3.889720`.)

## References

$[1]$ Harris, C.R., Millman, K.J., van der Walt, S.J. et al. (2020). Array programming with NumPy. *Nature* 585, 357-362.

$[2]$ Virtanen, P., Gommers, R., Oliphant, T.E. et al. (2020). SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python. *Nature Methods*, 17(3), 261-272.